# 04 · Modeling — Linear Regression (baseline) + XGBoost
**Data Science Diploma · ENES UNAM León**

### Evaluation Strategy for Time Series

> ⚠️ **Important:** In time series **random `train_test_split` is NOT used**. If you mix future dates into training, the model 'cheats' (data leakage). We always split chronologically:

```
|------- Train (70%) -------|-- Val (15%) --|-- Test (15%) --|
2024-01                  ~Oct 2024       ~Dec 2024       2025-03
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Load dataset with features
df = pd.read_csv('../data/processed/sol_usd_features.csv', index_col=0, parse_dates=True)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(f'Dataset loaded: {df.shape}')
df.head(3)

## 4.1 Define Features and Target

In [ ]:
FEATURE_COLS = [
    'Close_lag_1', 'Close_lag_2', 'Close_lag_3', 'Close_lag_5', 'Close_lag_7', 'Close_lag_14',
    'SMA_7', 'SMA_21', 'EMA_12', 'EMA_26',
    'RSI_14', 'MACD', 'MACD_signal', 'MACD_hist',
    'BB_upper', 'BB_lower', 'BB_width',
    'volatility_21d', 'Volume_change', 'Volume_SMA7',
    'day_of_week', 'month'
]
TARGET = 'Target'

X = df[FEATURE_COLS]
y = df[TARGET]

print(f'Features : {X.shape[1]}')
print(f'Samples  : {X.shape[0]}')
print(f'Target   : next day closing price (USD)')

## 4.2 Chronological Split (No Data Leakage)

In [ ]:
n = len(df)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train, y_train = X.iloc[:train_end],  y.iloc[:train_end]
X_val,   y_val   = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test,  y_test  = X.iloc[val_end:],    y.iloc[val_end:]

dates_train = df.index[:train_end]
dates_val   = df.index[train_end:val_end]
dates_test  = df.index[val_end:]

print(f'Train : {len(X_train)} samples | {dates_train[0].date()} → {dates_train[-1].date()}')
print(f'Val   : {len(X_val)}   samples | {dates_val[0].date()}   → {dates_val[-1].date()}')
print(f'Test  : {len(X_test)}  samples | {dates_test[0].date()}  → {dates_test[-1].date()}')

# Visualize the split
plt.figure(figsize=(14, 3))
plt.plot(dates_train, y_train, label='Train', color='#9945FF', linewidth=1.5)
plt.plot(dates_val,   y_val,   label='Validation', color='#FFB800', linewidth=1.5)
plt.plot(dates_test,  y_test,  label='Test', color='#14F195', linewidth=1.5)
plt.axvline(dates_val[0],  color='white', linestyle='--', alpha=0.5)
plt.axvline(dates_test[0], color='white', linestyle='--', alpha=0.5)
plt.title('Dataset Temporal Split')
plt.ylabel('USD')
plt.legend()
plt.tight_layout()
plt.savefig('../data/raw/04_split.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.3 Normalization

Linear regression requires features to be on the same scale. XGBoost does not require it, but applying it does no harm.

In [ ]:
scaler = StandardScaler()

# IMPORTANT: fit only on train, transform on val and test
# This prevents data leakage during normalization
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

print('Normalization applied ✓')
print(f'Feature mean (train): {X_train_sc.mean(axis=0).mean():.6f} (≈0 expected)')
print(f'Std dev (train)      : {X_train_sc.std(axis=0).mean():.4f} (≈1 expected)')

## 4.4 Metrics Function

In [ ]:
def evaluate_model(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2   = r2_score(y_true, y_pred)
    print(f'\n📊 {name}')
    print(f'  MAE  = ${mae:.2f}   (average absolute error)')
    print(f'  RMSE = ${rmse:.2f}  (penalizes large errors)')
    print(f'  MAPE = {mape:.2f}%  (average percentage error)')
    print(f'  R²   = {r2:.4f}    (1.0 = perfect)')
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}

results = []

## 4.5 Model 1: Linear Regression (Baseline)

The baseline is the minimum reference point. If XGBoost does not beat this simple model, something is wrong.

In [ ]:
lr = Ridge(alpha=1.0)
lr.fit(X_train_sc, y_train)

pred_lr_val  = lr.predict(X_val_sc)
pred_lr_test = lr.predict(X_test_sc)

res_lr_val  = evaluate_model('Ridge Regression (Validation)', y_val, pred_lr_val)
res_lr_test = evaluate_model('Ridge Regression (Test)', y_test, pred_lr_test)
results.append(res_lr_test)

In [ ]:
# Visualize predictions on test set
plt.figure(figsize=(14, 5))
plt.plot(dates_test, y_test.values, label='Actual', color='#9945FF', linewidth=1.5)
plt.plot(dates_test, pred_lr_test, label='Ridge Prediction', color='#FFB800', linewidth=1.5, linestyle='--')
plt.title('Ridge Regression — Prediction vs Actual (Test set)')
plt.ylabel('USD')
plt.legend()
plt.tight_layout()
plt.savefig('../data/raw/04_ridge_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.6 Model 2: XGBoost

Gradient Boosting based on decision trees. Widely used in data competitions and finance for its ability to capture non-linear relationships.

In [ ]:
# Training with early stopping using the validation set
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    early_stopping_rounds=30,
    eval_metric='rmse',
    verbosity=0
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print(f'Best iteration (early stopping): {xgb_model.best_iteration}')

In [ ]:
pred_xgb_val  = xgb_model.predict(X_val)
pred_xgb_test = xgb_model.predict(X_test)

res_xgb_val  = evaluate_model('XGBoost (Validation)', y_val, pred_xgb_val)
res_xgb_test = evaluate_model('XGBoost (Test)', y_test, pred_xgb_test)
results.append(res_xgb_test)

In [ ]:
# Visualize predictions on test set
plt.figure(figsize=(14, 5))
plt.plot(dates_test, y_test.values, label='Actual', color='#9945FF', linewidth=1.5)
plt.plot(dates_test, pred_xgb_test, label='XGBoost Prediction', color='#14F195', linewidth=1.5, linestyle='--')
plt.title('XGBoost — Prediction vs Actual (Test set)')
plt.ylabel('USD')
plt.legend()
plt.tight_layout()
plt.savefig('../data/raw/04_xgboost_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.7 Feature Importance (XGBoost)

In [ ]:
importance = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
colors = ['#9945FF' if i >= len(importance) - 5 else '#555' for i in range(len(importance))]
importance.plot(kind='barh', color=colors)
plt.title('XGBoost — Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('../data/raw/04_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 most important features:')
print(importance.tail(5).sort_values(ascending=False))

## 4.8 Final Model Comparison

In [ ]:
df_results = pd.DataFrame(results)
df_results.set_index('model', inplace=True)
print('\n===== MODEL COMPARISON (Test Set) =====')
print(df_results.round(4).to_string())

# Comparison chart
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric in zip(axes, ['MAE', 'RMSE', 'MAPE']):
    df_results[metric].plot(kind='bar', ax=ax, color=['#FFB800', '#14F195'], edgecolor='none')
    ax.set_title(metric)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
    ax.set_ylabel('USD' if metric != 'MAPE' else '%')

plt.suptitle('Metric Comparison — Test Set', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../data/raw/04_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Side-by-side predictions on test set
plt.figure(figsize=(14, 5))
plt.plot(dates_test, y_test.values,    label='Actual',  color='#9945FF', linewidth=2)
plt.plot(dates_test, pred_lr_test,     label='Ridge',   color='#FFB800', linewidth=1.5, linestyle='--', alpha=0.8)
plt.plot(dates_test, pred_xgb_test,    label='XGBoost', color='#14F195', linewidth=1.5, linestyle='--', alpha=0.8)
plt.title('Model Comparison — Test Set')
plt.ylabel('USD')
plt.legend()
plt.tight_layout()
plt.savefig('../data/raw/04_prediction_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.9 Save Models and Predictions

In [ ]:
import pickle, os
os.makedirs('../data/processed', exist_ok=True)

# Save models
with open('../data/processed/ridge_model.pkl', 'wb') as f:
    pickle.dump(lr, f)
with open('../data/processed/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
xgb_model.save_model('../data/processed/xgboost_model.json')

# Save predictions for the evaluation notebook
df_preds = pd.DataFrame({
    'date': dates_test,
    'real': y_test.values,
    'pred_ridge': pred_lr_test,
    'pred_xgb': pred_xgb_test
}).set_index('date')
df_preds.to_csv('../data/processed/predicciones_test.csv')

print('Models and predictions saved ✓')

---
## ✅ Modeling Summary

| Model | MAE | RMSE | MAPE | R² |
|---|---|---|---|---|
| Ridge Regression | *see output* | *see output* | *see output* | *see output* |
| XGBoost | *see output* | *see output* | *see output* | *see output* |

**Next step →** `05_evaluation.ipynb`